In [13]:
import numpy as np
import seaborn as sns # optional
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import matplotlib.image as mpimg
import pathlib

In [19]:
from pathlib import Path
from datetime import datetime
import numpy as np
import rasterio

#  1. Point at the data folder
here = Path("sentinel-data")          # change to an absolute Path(...) if needed
print("Looking in:", here.resolve())
print("Exists:", here.exists())
print([p.name for p in here.iterdir()] if here.exists() else "folder not found")


# 2. File discovery
def get_file_list(where, year):
    """Return all .tiff files in the S2_<year>-* folder(s) under `where`."""
    return sorted(where.glob(f"**/*{year}*/*.tiff"))


def get_band(filelist, band):
    """Pick a band file by name, e.g. get_band(files, 'B04')."""
    matches = [f for f in filelist if f"_{band}_" in f.name]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected 1 file for {band}, found {len(matches)}")
    return matches[0]


def folder_date(filelist):
    """Date from the folder name, e.g. S2_2017-07-12 -> datetime(2017, 7, 12)."""
    return datetime.strptime(filelist[0].parent.name, "S2_%Y-%m-%d")


Looking in: /Users/palakshi/Documents/GitHub/SDL-Python-Assignment/Assignment_2/Sentinel/sentinel-data
Exists: True
['S2_2017-07-12', 'S2_2018-07-12', 'White_light', 'S2_2019-07-12']


In [23]:
GE_images = here / "White_light"

def parse_date(path):
    return datetime.strptime(path.stem, "%Y%b")

ge_files = sorted(GE_images.glob("*.jpg"), key=parse_date)
print(ge_files)

[PosixPath('sentinel-data/White_light/2014Feb.jpg'), PosixPath('sentinel-data/White_light/2016May.jpg'), PosixPath('sentinel-data/White_light/2020Nov.jpg'), PosixPath('sentinel-data/White_light/2021Dec.jpg'), PosixPath('sentinel-data/White_light/2022Jan.jpg'), PosixPath('sentinel-data/White_light/2023Oct.jpg')]


In [25]:
files = sorted((here / "White_light").glob("*.jpg"))
wl_images = {f.stem: mpimg.imread(f) for f in files}

print(wl_images.keys())

dict_keys(['2014Feb', '2016May', '2020Nov', '2021Dec', '2022Jan', '2023Oct'])


In [20]:

# 3. Pixel area in m^2 from the metadata
def pixel_area_m2(path):
    with rasterio.open(path) as src:
        dx_deg, dy_deg = src.res
        b = src.bounds
    lat0 = 0.5 * (b.top + b.bottom)
    dy_m = dy_deg * 111_320
    dx_m = dx_deg * 111_320 * np.cos(np.radians(lat0))
    return dx_m * dy_m


# 4. NDVI
def read_band(path):
    with rasterio.open(path) as src:
        return src.read(1).astype("float32")


def compute_ndvi(filelist):
    red = read_band(get_band(filelist, "B04"))
    nir = read_band(get_band(filelist, "B08"))
    denom = nir + red
    return np.where(denom == 0, np.nan, (nir - red) / denom)


# 5. Run for every year
results = {}
for year in (2017, 2018, 2019):
    files = get_file_list(here, year)
    if not files:
        print(f"{year}: no files found")
        continue

    ndvi = compute_ndvi(files)
    area = pixel_area_m2(get_band(files, "B04"))
    veg_km2 = np.nansum(ndvi > 0.4) * area / 1e6

    results[year] = {"date": folder_date(files), "ndvi": ndvi,
                     "pixel_area_m2": area, "veg_km2": veg_km2}
    print(f"{year}: {len(files)} files, pixel area = {area:.1f} m^2, "
          f"vegetation = {veg_km2:.2f} km^2")

2017: 22 files, pixel area = 87.7 m^2, vegetation = 0.48 km^2
2018: 22 files, pixel area = 87.7 m^2, vegetation = 0.56 km^2
2019: 22 files, pixel area = 87.7 m^2, vegetation = 0.61 km^2


/var/folders/jd/hbc05gjd5cg61jrhc3ffmt480000gn/T/ipykernel_55507/3048706284.py:22: RuntimeWarning: invalid value encountered in divide
  return np.where(denom == 0, np.nan, (nir - red) / denom)
